# Task 3 — MAFixedwingDogfightEnvV2 with PPO (three-phase self-play)

**Goal.** Train a PPO policy for the 1v1 fixed-wing dogfight tournament using a
**three-phase self-play curriculum** (random → self-snapshot → league), then
submit two agents to the course tournament: the Phase-C end (league-trained)
and the Phase-B end (self-snapshot-trained) checkpoints.

**Pipeline.**

1. Configure the training driver (delegated to `training_cell_v3.py`).
2. Train $3$ independent seeds, each for $1\,000\,000$ env steps split as
   $200\text{k}$ (Phase A) $+$ $500\text{k}$ (Phase B, snapshot every $100$k) $+$
   $300\text{k}$ (Phase C, league of $5$ snapshots).
3. Plot the training-reward curve for each phase (diagnostic only — not the
   evaluation criterion; see below).
4. Run the provided `scripts/tournament.py` on the $3 \times 2 = 6$ final
   checkpoints (Phase-B end and Phase-C end, per seed), computing local Elo.
5. *(Optional, ~20 min)* Run a head-to-head probe of each seed's Phase-C end
   against its earliest and latest Phase-B snapshots, as a cycle-collapse diagnostic.
6. Generate the two tournament submission files.

**Note on evaluation.** The training-reward curve is **not** the tournament
evaluation criterion: the tournament uses `info["team_win"]` with a $\pm 50$
reward-margin fallback. We report Elo and head-to-head win-rates as the headline
numbers, and the training-reward curve as diagnostic context only.

**Compute.** ~$30$–$45$ minutes per seed × $3$ seeds ≈ $1.5$–$2$ hours total,
plus ~$10$ minutes for the round-robin tournament.

In [ ]:
!pip install stable_baselines3
!pip install PyFlyt
!pip install PyBullet
!pip install tqdm
!pip install matplotlib
!pip install numpy
!pip install torch

## 1. Imports and paths

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import torch

# PyFlyt env registration is done inside the training subprocess and inside
# the tournament script, not here — the notebook doesn't construct envs directly.
import stable_baselines3 as sb3
from stable_baselines3 import PPO   # for loading checkpoints if we need to introspect

# Make scripts/ importable (dogfight_wrapper.py, tournament.py, submission_template.py).
PROJECT_ROOT = Path.cwd().resolve()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

print(f"Stable-Baselines3 version: {sb3.__version__}")
print(f"PyTorch version:           {torch.__version__}")
print(f"CUDA available:            {torch.cuda.is_available()}")
print(f"Project root:              {PROJECT_ROOT}")
print(f"Scripts dir:               {SCRIPTS_DIR}")


## 2. Configuration

Hyperparameters are identical to Tasks 1 and 2 (`n_steps=2048`, since dogfight
episodes are at most $60\,\text{s} \times 30\,\text{Hz} = 1800$ steps, the long-rollout
argument from Waypoints does not apply here).

In [ ]:
# ---- Experiment identity ---------------------------------------------------
ALGO_NAME = "PPO"

# ---- Compute budget (REDUCED for 9h total wall-clock limit) ----------------
# Rationale: dogfight is computationally heavier than Hover/Waypoints
# (fixed-wing aerodynamics + opponent neural-net inference on every step).
# Realistic throughput is ~50-150 steps/s on 4 vec envs.  At 80 steps/s,
# 600k steps takes ~125 min, so 3 seeds fit in ~6.5h with margin.
SEEDS = [0, 1, 2]

PHASE_A_STEPS = 120_000                  # random opponent
PHASE_B_STEPS = 300_000                  # self-snapshot opponent
PHASE_C_STEPS = 180_000                  # league opponent
SNAPSHOT_FREQ = 75_000                   # within Phase B: 4 snapshots saved
                                          # (300k / 75k = 4 snapshots)
# Total per seed = 600,000 env steps.

N_ENVS = 4
TIMEOUT_PER_SEED = 180 * 60              # 3 h — generous given expected ~2h
PROGRESS_PRINT_EVERY = 20_000             # in-phase progress print cadence

# ---- PPO hyperparameters (same defaults as Tasks 1 & 2) -------------------
PPO_KWARGS = dict(
    policy="MlpPolicy",
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=dict(net_arch=dict(pi=[64, 64], vf=[64, 64])),
    verbose=0,
)

# ---- Output paths ---------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR  = RESULTS_DIR / "models"
LOGS_DIR    = RESULTS_DIR / "logs"
EVAL_DIR    = RESULTS_DIR / "eval"
FIGURES_DIR = RESULTS_DIR / "figures"
SUBMIT_DIR  = RESULTS_DIR / "submissions"
for d in (MODELS_DIR, LOGS_DIR, EVAL_DIR, FIGURES_DIR, SUBMIT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Group identity (used in submission filenames) ------------------------
GROUP_ID = "groupXX"                     # <-- TODO: replace with your group ID

# ---- Behaviour flags ------------------------------------------------------
FORCE_RETRAIN = True
RUN_WIN_RATE_PROBE = False

## 3. Run training (3 seeds × 3 phases)

Training is delegated to **`training_cell_v3.py`**, a self-contained module
that writes a standalone `_train_worker_dogfight.py` script to disk and runs
it once per seed via `subprocess.run([sys.executable, ...])`. Each subprocess
is a fresh Python interpreter, so PyBullet socket leaks die with the process
and the parent stays clean.

Per-seed flow inside the worker (preregistered in §7 of the report):

* **Phase A** — opponent is `action_space.sample()` (no learnt policy).
* **Phase B** — at every $100\,\text{k}$ step boundary, snapshot the current policy,
  reload it as the frozen opponent, and record its path.
* **Phase C** — opponent is sampled uniformly from the $5$ Phase-B snapshots
  at the start of every episode (via a `LeagueDogfightEnv` subclass).

**Before running this cell, save `training_cell_v3.py` next to this notebook**
(or anywhere on `sys.path`).

In [ ]:
import importlib
import training_cell_v3
from pathlib import Path
import sys

# 1. Update training_cell_v3.py to ensure 'import os' is INSIDE the worker TEMPLATE
with open('training_cell_v3.py', 'r') as f:
    content = f.read()

# Inject 'import os' into the worker script's TEMPLATE section
# We target the common import block used in the worker generation
if 'import os' not in content or content.count('import os') < 2:
    # Inject into worker TEMPLATE imports
    content = content.replace('import time\n    import torch', 'import time\n    import os\n    import torch')
    # Ensure global driver has it too
    if 'import os' not in content.split('\n')[:15]:
        content = content.replace('from __future__ import annotations', 'from __future__ import annotations\nimport os')

with open('training_cell_v3.py', 'w') as f:
    f.write(content)

# 2. Reload and initialize
importlib.reload(training_cell_v3)
from training_cell_v3 import DogfightTrainingConfig, setup_training_script, train_all

cfg = DogfightTrainingConfig(
    algo_name=ALGO_NAME,
    phase_a_steps=PHASE_A_STEPS,
    phase_b_steps=PHASE_B_STEPS,
    phase_c_steps=PHASE_C_STEPS,
    snapshot_freq=SNAPSHOT_FREQ,
    n_envs=N_ENVS,
    timeout_per_seed=TIMEOUT_PER_SEED,
    ppo_kwargs=PPO_KWARGS,
    scripts_dir=SCRIPTS_DIR, models_dir=MODELS_DIR, logs_dir=LOGS_DIR,
    project_root=PROJECT_ROOT,
    force_retrain=FORCE_RETRAIN,
)

# 3. Regenerate and run
setup_training_script(PROJECT_ROOT)
final_checkpoints, failed = train_all(SEEDS, cfg)

In [ ]:
with open('training_cell_v3.py', 'r') as f:
    print(f.read())

In [ ]:
with open('training_cell_v3.py', 'r') as f:
    print(f.read())

In [ ]:
import re

# Read the current content of the file
with open('training_cell_v3.py', 'r') as f:
    content = f.read()

# 1. Update the make_vec signature and body in the template string
old_make_vec = """    def make_vec(base_seed, league=False):
        fns = [make_env_thunk(i, base_seed, league=league) for i in range(args.n_envs)]
        return VecMonitor(DummyVecEnv(fns))"""

new_make_vec = """    def make_vec(base_seed, league=False, phase_tag='A'):
        fns = [make_env_thunk(i, base_seed, league=league) for i in range(args.n_envs)]
        log_subdir = os.path.join(args.log_path, f"phase_{phase_tag}")
        os.makedirs(log_subdir, exist_ok=True)
        return VecMonitor(DummyVecEnv(fns), filename=log_subdir)"""

content = content.replace(old_make_vec, new_make_vec)

# 2. Update the calls to make_vec in main()
content = content.replace('env_a = make_vec(args.seed)', 'env_a = make_vec(args.seed, phase_tag="A")')
content = content.replace('env_b = make_vec(args.seed + 1000000)', 'env_b = make_vec(args.seed + 1000000, phase_tag="B")')
content = content.replace('env_c = make_vec(args.seed + 2000000, league=True)', 'env_c = make_vec(args.seed + 2000000, league=True, phase_tag="C")')

# Write the corrected content back to disk
with open('training_cell_v3.py', 'w') as f:
    f.write(content)

print("Successfully patched training_cell_v3.py with correct logging logic.")

In [ ]:
with open('_train_worker_dogfight.py', 'r') as f:
    lines = f.readlines()
    last_20 = lines[-20:]
    print(''.join(last_20))

## 4. Training reward curves (diagnostic only)

We plot the SB3 episode-reward log from each phase, per seed. The signal we
expect to see is:

* **Phase A** climbing from a very negative value (the agent crashes immediately
  against random opponents) toward a baseline where it stays alive.
* **Phase B** non-monotonic — the reward drops after each snapshot refresh
  (the new opponent is harder than the previous) and recovers as the policy
  adapts.
* **Phase C** more volatile still, since the opponent resamples per-episode.

Reminder: this curve is **not** the tournament evaluation criterion. The
headline number is Elo, computed in cell 6.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from stable_baselines3.common.results_plotter import load_results, ts2xy

def load_monitor_rewards(log_path: Path):
    """Load the Monitor CSVs written by SB3 inside one seed's log directory."""
    try:
        df = load_results(str(log_path))
        if df.empty:
            return np.array([]), np.array([])
        ts, rews = ts2xy(df, "timesteps")
        return np.asarray(ts), np.asarray(rews)
    except Exception as e:
        return np.array([]), np.array([])

fig, axes = plt.subplots(1, len(SEEDS), figsize=(5 * len(SEEDS), 4), sharey=True)
if len(SEEDS) == 1:
    axes = [axes]

for ax, seed in zip(axes, SEEDS):
    log_path = cfg.log_path(seed)
    ts, rews = load_monitor_rewards(log_path)

    if len(ts) == 0:
        ax.set_title(f"seed {seed} \n(Waiting for data...)", fontsize=10)
        ax.text(0.5, 0.5, "Training in progress...\nor rerun required to\ngenerate monitor CSVs.",
                ha='center', va='center', transform=ax.transAxes, color='C1')
        continue

    ax.plot(ts, rews, alpha=0.4, lw=0.7, color="C0", label="episode reward")

    if len(rews) >= 50:
        window = 50
        ma = np.convolve(rews, np.ones(window) / window, mode="valid")
        ax.plot(ts[window - 1:], ma, lw=1.8, color="C3", label=f"{window}-ep MA")

    ax.axvline(PHASE_A_STEPS, color="gray", ls="--", lw=0.8, alpha=0.7)
    ax.axvline(PHASE_A_STEPS + PHASE_B_STEPS, color="gray", ls="--", lw=0.8, alpha=0.7)

    y_top = ax.get_ylim()[1] if len(rews) > 0 else 1.0
    ax.text(PHASE_A_STEPS / 2, y_top * 0.9, "A", ha="center")
    ax.text(PHASE_A_STEPS + PHASE_B_STEPS / 2, y_top * 0.9, "B", ha="center")
    ax.text(PHASE_A_STEPS + PHASE_B_STEPS + PHASE_C_STEPS / 2, y_top * 0.9, "C", ha="center")

    ax.set_xlabel("Steps")
    ax.set_title(f"seed {seed}")
    ax.grid(alpha=0.3)
    if seed == SEEDS[0]:
        ax.set_ylabel("Reward")
        ax.legend(loc="lower right", fontsize=8)

fig.suptitle(f"{ALGO_NAME} Self-Play Training Progress (Diagnostic)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Local round-robin tournament (the headline metric)

We call the provided `scripts/tournament.py` directly on our $3 \times 2 = 6$
final checkpoints (Phase-B end and Phase-C end, per seed). Each pair plays 5
matches; ratings update via the standard Elo formula with $K = 32$ and initial
rating $1500$, exactly as in the grading script.

The output is a JSON file containing the ranked list of checkpoints by Elo,
plus the full match history. The top Elo from each phase becomes a submission
candidate.

In [ ]:
# Build the list of candidate checkpoints (only seeds that completed training).
candidate_paths: list[Path] = []
for seed in SEEDS:
    for phase in ("B", "C"):
        ckpt = cfg.model_path(seed, phase).with_suffix(".zip")
        if ckpt.exists():
            candidate_paths.append(ckpt)
        else:
            print(f"[warn] missing checkpoint: {ckpt}")

print(f"\nFound {len(candidate_paths)} candidates for the tournament:")
for p in candidate_paths:
    print(f"  {p.name}")


In [ ]:
# Import tournament.py and call run_tournament directly (no shell subprocess).
from tournament import run_tournament

tournament_results = run_tournament(
    [str(p) for p in candidate_paths],
    matches_per_pair=5,
    render=False,
)

# Persist for later analysis / report figures.
tournament_json = EVAL_DIR / "local_tournament.json"
with open(tournament_json, "w") as f:
    json.dump(tournament_results, f, indent=2, default=str)
print(f"\nTournament results saved to {tournament_json}")


In [ ]:
# Print the rankings in a compact format and identify the two submission candidates.
rankings = tournament_results["rankings"]

print("=" * 60)
print("FINAL LOCAL RANKINGS")
print("=" * 60)
print(f"{'Rank':<6}{'Checkpoint':<50}{'Elo':>8}")
print("-" * 60)
for r in rankings:
    print(f"  {r['rank']:<4}{r['name']:<48}{r['elo']:>8.1f}")
print("=" * 60)

# Identify top Phase-B and top Phase-C checkpoints (our two submission candidates).
def first_with(phase_tag: str):
    for r in rankings:
        if f"_phase{phase_tag}" in r["name"]:
            return r
    return None

top_b = first_with("B")
top_c = first_with("C")
print(f"\nTop Phase-B candidate (selfplay submission): {top_b['name']}  Elo={top_b['elo']:.1f}")
print(f"Top Phase-C candidate (league   submission): {top_c['name']}  Elo={top_c['elo']:.1f}")


## 6. Optional: cycle-collapse probe via win-rate against past snapshots

For each seed, we play 3 matches of the Phase-C end checkpoint against the
**earliest** Phase-B snapshot and 3 against the **latest** Phase-B snapshot.
If the Phase-C policy wins against both with similar rates, the league phase
is doing its job. If it wins against the latest but *loses* against the earliest,
we have cycle collapse: the policy has overwritten its previous capabilities.

This adds ~$20$ minutes to the notebook runtime. Toggle `RUN_WIN_RATE_PROBE`
in cell 4 to enable.

In [ ]:
def _play_match(model_a_path: Path, model_b_path: Path, n_games: int = 3,
                seed_offset: int = 0):
    """Run n_games between two SB3 checkpoints; return (wins_a, wins_b, draws).

    Mirrors the win-detection logic of scripts/tournament.py.run_match() so that
    the numbers are comparable to the round-robin Elo.
    """
    from PyFlyt.pz_envs import MAFixedwingDogfightEnvV2

    model_a = PPO.load(str(model_a_path))
    model_b = PPO.load(str(model_b_path))
    wins_a, wins_b, draws = 0, 0, 0

    for g in range(n_games):
        env = MAFixedwingDogfightEnvV2(
            team_size=1, assisted_flight=True, flatten_observation=True,
            render_mode=None, max_duration_seconds=60.0, agent_hz=30,
        )
        observations, _ = env.reset(seed=seed_offset + g)
        rewards_acc = {a: 0.0 for a in env.agents}

        for _ in range(1800):
            actions = {}
            for agent in env.agents:
                policy = model_a if agent == "uav_0" else model_b
                action, _ = policy.predict(observations[agent], deterministic=True)
                actions[agent] = action
            observations, rewards, terminations, truncations, infos = env.step(actions)
            for a in rewards:
                rewards_acc[a] = rewards_acc.get(a, 0.0) + rewards.get(a, 0.0)
            done = all(terminations.get(a, True) or truncations.get(a, False)
                       for a in env.agents) or len(env.agents) == 0
            if done:
                break
        try:
            env.close()
        except Exception:
            pass

        # team_win flag has priority; fallback to ±50 reward margin.
        a_won = b_won = False
        for agent, info in infos.items():
            if info.get("team_win", False):
                if agent == "uav_0":
                    a_won = True
                else:
                    b_won = True
        if not a_won and not b_won:
            r_a, r_b = rewards_acc.get("uav_0", 0.0), rewards_acc.get("uav_1", 0.0)
            if r_a > r_b + 50: a_won = True
            elif r_b > r_a + 50: b_won = True

        if a_won and not b_won: wins_a += 1
        elif b_won and not a_won: wins_b += 1
        else: draws += 1

    return wins_a, wins_b, draws


if RUN_WIN_RATE_PROBE:
    win_rate_results = {}
    for seed in SEEDS:
        snap_json = cfg.snapshot_dir(seed) / "snapshots.json"
        if not snap_json.exists():
            print(f"[skip] seed {seed}: no snapshots.json")
            continue
        with open(snap_json) as f:
            snapshots = [Path(p) for p in json.load(f)]
        if not snapshots:
            print(f"[skip] seed {seed}: empty snapshot list")
            continue

        phase_c_end = cfg.model_path(seed, "C").with_suffix(".zip")
        if not phase_c_end.exists():
            print(f"[skip] seed {seed}: no Phase-C checkpoint")
            continue

        first_snap, last_snap = snapshots[0], snapshots[-1]
        print(f"\n--- seed {seed} ---")
        wA, lA, dA = _play_match(phase_c_end, first_snap, n_games=3, seed_offset=seed * 1000)
        print(f"  Phase-C vs FIRST snapshot ({first_snap.name}): "
              f"{wA}W/{lA}L/{dA}D")
        wB, lB, dB = _play_match(phase_c_end, last_snap,  n_games=3, seed_offset=seed * 1000 + 100)
        print(f"  Phase-C vs LAST  snapshot ({last_snap.name}):  "
              f"{wB}W/{lB}L/{dB}D")
        win_rate_results[seed] = {
            "vs_first_snapshot": dict(wins=wA, losses=lA, draws=dA, name=first_snap.name),
            "vs_last_snapshot":  dict(wins=wB, losses=lB, draws=dB, name=last_snap.name),
        }

    out = EVAL_DIR / "win_rate_probe.json"
    with open(out, "w") as f:
        json.dump(win_rate_results, f, indent=2)
    print(f"\nSaved win-rate probe to {out}")
else:
    print("Win-rate probe skipped (RUN_WIN_RATE_PROBE=False).")


## 7. Generate tournament submission files

We create two submission files following `scripts/submission_template.py`:

* `{GROUP_ID}_league.py`  — wraps the top Phase-C checkpoint (primary bet).
* `{GROUP_ID}_selfplay.py` — wraps the top Phase-B checkpoint (safer fallback).

Each `.py` file is bundled with the corresponding `.zip` weights in
`results/submissions/`. The `load_model()` function in each `.py` resolves the
weights path via `os.path.dirname(__file__)`, so the bundle is self-contained.

In [ ]:
import shutil

SUBMISSION_TEMPLATE = """\
\"\"\"Tournament submission for {group_id} ({label}).

Loads a Stable-Baselines3 PPO checkpoint trained via the three-phase self-play
curriculum described in Section 7 of the report. The weights file
({weights_filename}) is bundled in the same directory as this script.
\"\"\"

import os
from stable_baselines3 import PPO


def load_model(path=None):
    \"\"\"Return a trained model with .predict(obs, deterministic=True).\"\"\"
    if path is None:
        here = os.path.dirname(os.path.abspath(__file__))
        path = os.path.join(here, "{weights_filename}")
    return PPO.load(path)
"""


def make_submission(checkpoint_zip: Path, label: str) -> tuple[Path, Path]:
    """Bundle a .py shim and the .zip checkpoint into SUBMIT_DIR/{group}_{label}.*"""
    py_name  = f"{GROUP_ID}_{label}.py"
    zip_name = f"{GROUP_ID}_{label}.zip"
    out_py   = SUBMIT_DIR / py_name
    out_zip  = SUBMIT_DIR / zip_name

    out_py.write_text(SUBMISSION_TEMPLATE.format(
        group_id=GROUP_ID, label=label, weights_filename=zip_name))
    shutil.copyfile(checkpoint_zip, out_zip)
    return out_py, out_zip


# Resolve checkpoint paths from the tournament rankings.
def _ranking_to_path(ranking_name: str) -> Path:
    # The tournament returns names like "final_PPO_Dogfight_seed0_phaseC".
    # Strip a trailing ".zip" if present, then resolve.
    name = ranking_name.replace(".zip", "")
    return MODELS_DIR / f"{name}.zip"


league_ckpt   = _ranking_to_path(top_c["name"])
selfplay_ckpt = _ranking_to_path(top_b["name"])

print(f"Bundling submissions in {SUBMIT_DIR}:")
for ckpt, label in [(league_ckpt, "league"), (selfplay_ckpt, "selfplay")]:
    if not ckpt.exists():
        print(f"  [missing] {ckpt}")
        continue
    py, zp = make_submission(ckpt, label)
    print(f"  {py.name}  +  {zp.name}  (source: {ckpt.name})")

# Smoke-test: re-load each submission via its load_model() to catch packaging bugs.
print("\nSmoke-testing submissions:")
for label in ("league", "selfplay"):
    py = SUBMIT_DIR / f"{GROUP_ID}_{label}.py"
    if not py.exists():
        continue
    import importlib.util
    spec = importlib.util.spec_from_file_location(f"_submission_{label}", py)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    model = mod.load_model()
    dummy_obs = np.zeros(37, dtype=np.float32)
    action, _ = model.predict(dummy_obs, deterministic=True)
    assert action.shape == (4,), f"action shape {action.shape} != (4,)"
    print(f"  {py.name}: predict() returns action shape {action.shape}  OK")


## 8. Outputs

After running this notebook end-to-end you will have, per seed:

* `results/models/final_PPO_Dogfight_seed{0,1,2}_phase{A,B,C}.zip` — phase checkpoints.
* `results/models/snapshots_PPO_Dogfight_seed{0,1,2}/snapshot_*.zip` — Phase-B snapshots.
* `results/models/snapshots_PPO_Dogfight_seed{0,1,2}/snapshots.json` — snapshot index.
* `results/logs/PPO_Dogfight_seed{0,1,2}/` — SB3 tensorboard logs (one subdir per phase).

And once across seeds:

* `results/eval/local_tournament.json` — Elo rankings + full match history.
* `results/eval/win_rate_probe.json` — (optional) cycle-collapse diagnostic.
* `results/figures/dogfight_training_reward.png` — per-seed reward trace.
* `results/submissions/{GROUP_ID}_{league,selfplay}.{py,zip}` — the two
  files to upload for the tournament submission.

**To submit**: upload the four files in `results/submissions/` (two `.py` and
two `.zip`) to the Montefiore platform. Each `.py` is self-contained and
resolves its own weights file via `os.path.dirname(__file__)`.

## Diagnostic test on an early model and vizualisation

In [ ]:
from tournament import run_match
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

BEST_AGENT = PROJECT_ROOT / "results" / "models" / "final_PPO_Dogfight_seed2_phaseC.zip"

# There are no saved Phase-A snapshots, so this uses the earliest saved snapshot instead.
EARLY_CHECKPOINT = (
    PROJECT_ROOT
    / "results"
    / "models"
    / "snapshots_PPO_Dogfight_seed2"
    / "snapshot_000075000.zip"
)

# If you prefer comparing against the end of Phase A instead, use:
# EARLY_CHECKPOINT = PROJECT_ROOT / "results" / "models" / "final_PPO_Dogfight_seed2_phaseA.zip"

best_model = PPO.load(str(BEST_AGENT))
early_model = PPO.load(str(EARLY_CHECKPOINT))

result = run_match(
    best_model,
    early_model,
    n_games=9,
    max_steps=1800,
    seed_offset=100,
)

print(f"Best agent:        {BEST_AGENT.name}")
print(f"Reference model:   {EARLY_CHECKPOINT.name}")
print(f"Wins A / Wins B / Draws = {result['wins_a']} / {result['wins_b']} / {result['draws']}")
print(f"Mean rewards A / B      = {result['mean_reward_a']:.1f} / {result['mean_reward_b']:.1f}")

display(pd.DataFrame(result["games"]))

In [ ]:
from pathlib import Path
from stable_baselines3 import PPO
from tournament import run_match

PROJECT_ROOT = Path.cwd().resolve()

BEST_AGENT = PROJECT_ROOT / "results" / "models" / "final_PPO_Dogfight_seed2_phaseC.zip"
OPPONENT = PROJECT_ROOT / "results" / "models" / "final_PPO_Dogfight_seed2_phaseC.zip"

best_model = PPO.load(str(BEST_AGENT))
opponent_model = PPO.load(str(OPPONENT))

viz_result = run_match(
    best_model,
    opponent_model,
    n_games=1,
    max_steps=1800,
    render_mode="human",   # opens the PyFlyt/PyBullet window
    seed_offset=500,
)

print(viz_result)